# C001: Full Train Candidate Pool V1

**Objective**: Generate the final, frozen Candidate Pool V1 for the entire 2.2M Source 1 dataset using the proven R001 architecture.

**Rules**:
- Uses GPU TF-IDF with `batch_size=100`.
- Checkpoints every 50K rows per country to prevent data loss on OOM/Timeout.
- Generates a final packaged tarball.

## 1. Environment & GPU Verification

In [ ]:
import sys
import os
import subprocess

print(f"Python Version: {sys.version}")
print(f"CWD: {os.getcwd()}")
!nvidia-smi

try:
    import cuml
    import cupy as cp
    print("cuML/CuPy is available. GPU_BACKEND_ACTIVE = TRUE")
except ImportError:
    print("WARNING: cuML/CuPy not found. The script will halt loudly.")


## 2. Clone Repository

In [ ]:
REPO_URL = "https://github.com/yugtheguy/amazon_ml.git"
REPO_DIR = "/kaggle/working/amazon_ml"
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)


## 3. Link Processed Data & Folds

In [ ]:
PROCESSED_DATA_ROOT = None
known_path = "/kaggle/input/datasets/yugdeshmukh/amazon-ml-processed-v001"
if os.path.exists(os.path.join(known_path, "train_source1.parquet")):
    PROCESSED_DATA_ROOT = known_path
else:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train_source1.parquet" in files:
            PROCESSED_DATA_ROOT = root
            break
if PROCESSED_DATA_ROOT:
    print(f"Found processed dataset at {PROCESSED_DATA_ROOT}")
else:
    raise FileNotFoundError("CRITICAL: Processed data (train_source1.parquet) not found in /kaggle/input.")

RAW_DATA_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train_source1.tsv" in files and "train" in root:
        RAW_DATA_ROOT = os.path.dirname(root)
        break
if not RAW_DATA_ROOT:
    raise FileNotFoundError("CRITICAL: Raw data (train_source1.tsv) not found in /kaggle/input.")
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/train") and os.path.exists(os.path.join(RAW_DATA_ROOT, "train")):
    os.symlink(os.path.join(RAW_DATA_ROOT, "train"), "data/raw/train")
if not os.path.exists("data/raw/test") and os.path.exists(os.path.join(RAW_DATA_ROOT, "test")):
    os.symlink(os.path.join(RAW_DATA_ROOT, "test"), "data/raw/test")


## 4. Install Dependencies

In [ ]:
!pip install -r requirements.txt -q


## 5. Engineering Smoke Check (Small Pre-flight)

In [ ]:
import sys
import subprocess
import shutil
print("Running smoke test (1000 rows, chunk_size=250)...")
FOLD_MANIFEST = "/kaggle/working/amazon_ml/artifacts/folds/folds_v1.parquet"
if not os.path.exists(FOLD_MANIFEST):
    print(f"Generating fold manifest at {FOLD_MANIFEST}...")
    subprocess.run([sys.executable, "-u", "scripts/build_folds.py"], check=True, env={**os.environ, 'PYTHONPATH': '.'})
cmd = [
    sys.executable, "-u", "scripts/run_c001_full_train.py",
    "--data-dir", "data",
    "--processed-dir", PROCESSED_DATA_ROOT,
    "--fold-manifest", FOLD_MANIFEST,
    "--out-dir", "/kaggle/working/artifacts/candidate_pool",
    "--smoke-size", "1000",
    "--chunk-size", "250"
]
env = os.environ.copy()
env['PYTHONPATH'] = "."
try:
    subprocess.run(cmd, env=env, check=True)
    shutil.rmtree("/kaggle/working/artifacts/candidate_pool/C001")
    print("SMOKE PASSED")
except subprocess.CalledProcessError as e:
    print("SMOKE FAILED")
    raise e


## 6. C001 FULL TRAINING RUN (2.2M Rows)

In [ ]:
print("Starting FULL C001 Execution...")
cmd = [
    sys.executable, "-u", "scripts/run_c001_full_train.py",
    "--data-dir", "data",
    "--processed-dir", PROCESSED_DATA_ROOT,
    "--fold-manifest", FOLD_MANIFEST,
    "--out-dir", "/kaggle/working/artifacts/candidate_pool"
]
try:
    subprocess.run(cmd, env=env, check=True)
    print("FULL RUN COMPLETED")
except subprocess.CalledProcessError as e:
    print("FULL RUN FAILED")
    raise e


## 7. Metrics Review

In [ ]:
import json
import os
metrics_file = "/kaggle/working/artifacts/candidate_pool/C001/candidate_pool_v1/metrics/evaluation_metrics.json"
if os.path.exists(metrics_file):
    with open(metrics_file) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("Metrics file not found. Did the run complete?")


## 8. Extract Final Artifacts

In [ ]:
import shutil
import os
bundle_src = "/kaggle/working/artifacts/candidate_pool/C001/candidate_pool_v1/candidate_pool_v1_bundle.tar.gz"
bundle_dst = "/kaggle/working/candidate_pool_v1_bundle.tar.gz"
if os.path.exists(bundle_src):
    shutil.copy2(bundle_src, bundle_dst)
    print(f"Bundle ready at {bundle_dst}")
else:
    print("Bundle not found.")
